# PublicationAndPopularityInTopNovels

In this notebook, I explore how the Goodreads Top Ranked Novels dataset and the New York Times bestseller dataset represent different kinds of literary popularity. In particular, I look at publication trends, genre concentration, Goodreads ratings.

My main questions are:
- Which publication decades appear most often in the top novels dataset?
- Which genres are most common among top-ranked novels?
- Do some genres tend to receive higher Goodreads ratings than others?
- Which books remain on the New York Times bestseller list the longest?

## Imports and Data Loading

In [7]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [8]:
novelsdf = pd.read_csv(
    "https://raw.githubusercontent.com/melaniewalsh/responsible-datasets-in-context/main/datasets/top-500-novels/final_merged_dataset_no_full_text.tsv", 
    sep="\t", 
    encoding='utf-8'
)

nytdf = pd.read_csv(
    "https://raw.githubusercontent.com/ecds/post45-datasets/main/nyt_full.tsv",
    sep="\t",
    encoding="utf-8"
)

In [9]:
novelsdf.head()

,top_500_rank,title,author,pub_year,orig_lang,genre,author_birth,author_death,author_gender,author_primary_lang,...,gr_num_ratings,gr_num_reviews,gr_avg_rating_rank,gr_num_ratings_rank,oclc_owi,author_viaf,gr_url,wiki_url,pg_eng_url,pg_orig_url
0,1,Don Quixote,Miguel de Cervantes,1605,Spanish,action,1547,1616,male,spa,...,"269,435","12,053",318,211,1.810748e+09,17220427,https://www.goodreads.com/book/show/3836.Don_Q...,https://en.wikipedia.org/wiki/Don_Quixote,https://www.gutenberg.org/cache/epub/996/pg996...,https://www.gutenberg.org/cache/epub/2000/pg20...
1,2,Alice's Adventures in Wonderland,Lewis Carroll,1865,English,fantasy,1832,1898,male,eng,...,"561,016","15,380",172,133,1.156132e+10,66462036,https://www.goodreads.com/book/show/24213.Alic...,https://en.wikipedia.org/wiki/Alice%27s_Advent...,https://www.gutenberg.org/cache/epub/11/pg11.txt,NaN
2,3,The Adventures of Huckleberry Finn,Mark Twain,1884,English,action,1835,1910,male,eng,...,"1,262,480","19,440",373,68,3.373178e+09,50566653,https://www.goodreads.com/book/show/2956.The_A...,https://en.wikipedia.org/wiki/Adventures_of_Hu...,https://www.gutenberg.org/cache/epub/76/pg76.txt,NaN
3,4,The Adventures of Tom Sawyer,Mark Twain,1876,English,action,1835,1910,male,eng,...,"931,898","13,603",301,88,3.373178e+09,50566653,https://www.goodreads.com/book/show/24583.The_...,https://en.wikipedia.org/wiki/The_Adventures_o...,https://www.gutenberg.org/cache/epub/74/pg74.txt,NaN
4,5,Treasure Island,Robert Louis Stevenson,1883,English,action,1850,1894,male,eng,...,"486,155","16,307",368,145,3.434000e+03,95207986,https://www.goodreads.com/book/show/295.Treasu...,https://en.wikipedia.org/wiki/Treasure_Island,https://www.gutenberg.org/cache/epub/120/pg120...,NaN


In [10]:
nytdf.head()

,year,week,rank,title_id,title,author
0,1931,1931-10-12,1,6477,THE TEN COMMANDMENTS,Warwick Deeping
1,1931,1931-10-12,2,1808,FINCHE'S FORTUNE,Mazo de la Roche
2,1931,1931-10-12,3,5304,THE GOOD EARTH,Pearl S. Buck
3,1931,1931-10-12,4,4038,SHADOWS ON THE ROCK,Willa Cather
4,1931,1931-10-12,5,3946,SCARMOUCHE THE KING MAKER,Rafael Sabatini


## Understanding the Data

Before making visualizations, I want to understand what each dataset contains, how large it is, and what kinds of variables I can use for exploration.

In [11]:
print("Top 500 Novels Shape:", novelsdf.shape)
print("NYT Bestsellers Shape:", nytdf.shape)

Top 500 Novels Shape: (500, 29)
NYT Bestsellers Shape: (60386, 6)


In [12]:
novelsdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   top_500_rank              500 non-null    int64  
 1   title                     500 non-null    object 
 2   author                    500 non-null    object 
 3   pub_year                  500 non-null    int64  
 4   orig_lang                 500 non-null    object 
 5   genre                     500 non-null    object 
 6   author_birth              499 non-null    object 
 7   author_death              496 non-null    object 
 8   author_gender             500 non-null    object 
 9   author_primary_lang       500 non-null    object 
 10  author_nationality        500 non-null    object 
 11  author_field_of_activity  329 non-null    object 
 12  author_occupation         459 non-null    object 
 13  oclc_holdings             495 non-null    float64
 14  oclc_ehold

In [13]:
nytdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60386 entries, 0 to 60385
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   year      60386 non-null  int64 
 1   week      60386 non-null  object
 2   rank      60386 non-null  int64 
 3   title_id  60386 non-null  int64 
 4   title     60386 non-null  object
 5   author    60376 non-null  object
dtypes: int64(3), object(3)
memory usage: 2.8+ MB


## Missing Values

Checking for missing values helps clarify which columns are reliable for analysis and which ones may need to be handled more carefully.

In [14]:
novelsdf.isna().sum()

top_500_rank                  0
title                         0
author                        0
pub_year                      0
orig_lang                     0
genre                         0
author_birth                  1
author_death                  4
author_gender                 0
author_primary_lang           0
author_nationality            0
author_field_of_activity    171
author_occupation            41
oclc_holdings                 5
oclc_eholdings                5
oclc_total_editions           5
oclc_holdings_rank            5
oclc_editions_rank            5
gr_avg_rating                 0
gr_num_ratings                0
gr_num_reviews                0
gr_avg_rating_rank            0
gr_num_ratings_rank           0
oclc_owi                      5
author_viaf                   0
gr_url                       20
wiki_url                      0
pg_eng_url                    0
pg_orig_url                 436
dtype: int64

In [15]:
nytdf.isna().sum()


year         0
week         0
rank         0
title_id     0
title        0
author      10
dtype: int64

The Goodreads-style novels dataset has a few missing values in metadata columns, but the columns most useful for this analysis, such as title, author, publication year, genre, and Goodreads average rating, are mostly complete. The NYT dataset is also fairly clean, although a small number of author values are missing.

## Preparing the Data

To make the visualizations easier, I need to clean a few columns and create some new ones. In particular, I want to turn publication years into decades and convert Goodreads rating counts into numeric values.

In [21]:
novelsdf['pub_year'] = pd.to_numeric(novelsdf['pub_year'], errors='coerce')
novelsdf['gr_avg_rating'] = pd.to_numeric(novelsdf['gr_avg_rating'], errors='coerce')

novelsdf['grnumratings_clean'] = (
    novelsdf['gr_num_ratings']
    .astype(str)
    .str.replace(',', '', regex=False)
)
novelsdf['grnumratings_clean'] = pd.to_numeric(novelsdf['grnumratings_clean'], errors='coerce')

novelsdf['decade'] = (novelsdf['pub_year'] // 10) * 10
nytdf['year'] = pd.to_numeric(nytdf['year'], errors='coerce')

In [23]:
novelsdf[['title', 'pub_year', 'decade', 'genre', 'gr_avg_rating', 'gr_num_ratings']].head()

,title,pub_year,decade,genre,gr_avg_rating,gr_num_ratings
0,Don Quixote,1605,1600,action,3.90,"269,435"
1,Alice's Adventures in Wonderland,1865,1860,fantasy,4.06,"561,016"
2,The Adventures of Huckleberry Finn,1884,1880,action,3.83,"1,262,480"
3,The Adventures of Tom Sawyer,1876,1870,action,3.92,"931,898"
4,Treasure Island,1883,1880,action,3.84,"486,155"


## Publication Patterns in Top Novels

I first want to see whether the top novels dataset is concentrated in certain historical periods. Grouping books by decade makes it easier to see broad patterns rather than focusing on individual years.

In [24]:
decade_counts = (
    novelsdf.dropna(subset=['decade'])
    .groupby('decade')
    .size()
    .reset_index(name='count')
    .sort_values('decade')
)

decade_counts

,decade,count
0,1020,1
1,1350,1
2,1480,1
3,1510,1
4,1600,1
5,1670,2
6,1710,1
7,1720,2
8,1740,1
9,1750,2


In [25]:
alt.Chart(decade_counts).mark_bar().encode(
    x=alt.X('decade:O', title='Publication Decade'),
    y=alt.Y('count:Q', title='Number of Books'),
    tooltip=['decade', 'count']
).properties(
    title='Top Novels by Publication Decade',
    width=600,
    height=400
)

alt.Chart(...)

This chart shows which decades are most represented in the top novels dataset. 

## Genre Distribution

Next, I want to see which genres appear most often. This gives a quick sense of how broad or narrow the dataset is in terms of literary categories.

In [26]:
genre_counts = (
    novelsdf.dropna(subset=['genre'])
    .groupby('genre')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

genre_counts

,genre,count
8,na,221
5,history,53
4,fantasy,48
10,romance,33
3,bildung,27
11,scifi,21
12,thrillers,21
7,mystery,18
0,action,16
9,political,15


In [27]:
alt.Chart(genre_counts).mark_bar().encode(
    x=alt.X('count:Q', title='Number of Books'),
    y=alt.Y('genre:N', sort='-x', title='Genre'),
    tooltip=['genre', 'count']
).properties(
    title='Genre Distribution in Top Ranked Novels',
    width=600,
    height=400
)

alt.Chart(...)

This graph helps show the distribution of genres among the top-ranked novels, highlighting which genres are most prevalent in the dataset. It provides a visual representation of the genre composition in the dataset, allowing for a deeper understanding of the literary landscape represented by these novels. It may also reveal that some genre labels are broad or inconsistently used, which is important to remember when interpreting the results.

## Genre Across Decades

Genre counts alone are helpful, but it is also useful to see whether certain genres cluster in certain publication periods.

In [28]:
genre_decade = (
    novelsdf.dropna(subset=['genre', 'decade'])
    .groupby(['decade', 'genre'])
    .size()
    .reset_index(name='count')
)

genre_decade = genre_decade[genre_decade['genre'] != 'na']
genre_decade.head()

,decade,genre,count
4,1600,action,1
5,1670,allegories,1
7,1710,action,1
8,1720,fantasy,1
13,1770,autobio,1


In [29]:
alt.Chart(genre_decade).mark_bar().encode(
    x=alt.X('count:Q', title='Number of Books'),
    y=alt.Y('genre:N', sort='-x', title='Genre'),
    color=alt.Color('decade:O', title='Publication Decade'),
    tooltip=['decade', 'genre', 'count']
).properties(
    title='Genre Distribution Across Publication Decades',
    width=650,
    height=400
)

alt.Chart(...)

This chart shows how genre representation changes across publication decades. It can help identify whether some genres are associated more strongly with older or newer books in the dataset.

## Goodreads Ratings by Genre

After looking at frequency, I want to look at evaluation. Goodreads ratings provide one measure of reader response, even though they do not represent all readers equally.

In [31]:
genre_ratings = novelsdf.dropna(subset=['genre', 'gr_avg_rating']).copy()
genre_ratings = genre_ratings[genre_ratings['genre'] != 'na']

genre_ratings[['genre', 'gr_avg_rating']].head()

,genre,gr_avg_rating
0,action,3.90
1,fantasy,4.06
2,action,3.83
3,action,3.92
4,action,3.84


In [33]:
alt.Chart(genre_ratings).mark_boxplot().encode(
    x=alt.X('gr_avg_rating:Q', title='Goodreads Average Rating', scale=alt.Scale(zero=False)),
    y=alt.Y('genre:N', sort='-x', title='Genre'),
    color=alt.Color('genre:N', legend=None),
    tooltip=['genre', alt.Tooltip('mean(gr_avg_rating):Q', format='.2f')]
).properties(
    title='Goodreads Ratings by Genre',
    width=650,
    height=400
)

alt.Chart(...)

This visualization shows how Goodreads ratings vary across genres rather than just how often genres appear. Because ratings tend to fall within a narrow range, a boxplot is useful for showing spread and outliers instead of relying only on averages.

## Final Reflections

This exploratory analysis suggests that the top novels dataset and the NYT bestseller dataset measure different kinds of literary success. The Goodreads-style dataset is useful for studying reader ratings, genre groupings, and long-term popularity, while the NYT dataset highlights shorter-term bestseller visibility and repeated market presence.

There are also important limitations. Genre labels are uneven, Goodreads users do not represent all readers, and NYT bestseller appearances are not the same as literary quality. Even so, comparing the datasets helps reveal how “popular” books can mean different things depending on the platform, ranking system, and audience being measured.